# 04 — Kesimpulan

Notebook ini hanya membaca hasil notebook 02 dan 03. Tidak ada model yang
dilatih ulang. Tujuannya:

1. memverifikasi sidik jari data yang sama,
2. menampilkan tabel hold-out dan CV,
3. menyebut pemenang beserta alasan yang dihitung kode,
4. merangkum keputusan fitur usia,
5. membandingkan hasil dengan tiga jurnal,
6. menulis `kesimpulan.json`.

In [1]:
import importlib.util
import sys
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd()
module_path = cwd / "perbandingan_common.py"
if not module_path.exists():
    docs_module = cwd / "docs" / "perbandingan_common.py"
    if docs_module.exists():
        sys.path.insert(0, str(docs_module.parent))
    else:
        url = "https://raw.githubusercontent.com/RobbyCatur/rumahpoint/main/docs/perbandingan_common.py"
        print("Mengunduh modul bersama dari GitHub ...")
        urllib.request.urlretrieve(url, module_path)
        sys.path.insert(0, str(cwd))
else:
    sys.path.insert(0, str(cwd))

import perbandingan_common as c

print("modul:", c.__file__)
print("repo terdeteksi:", c.REPO_ROOT)
print("direktori keluaran:", c.OUTPUT_DIR)

modul: /opt/rumahpoint/docs/perbandingan_common.py
repo terdeteksi: /opt/rumahpoint
direktori keluaran: /opt/rumahpoint/docs/perbandingan_output


## 1. Verifikasi sidik jari data

Data dimuat kembali dan hash baris latih/uji dibandingkan dengan split yang
tersimpan. Jika ada yang tidak cocok, notebook berhenti; hasil tidak boleh
dibaca dari data yang berbeda.

In [2]:
df, rep, split, train_idx, test_idx, folds = c.load_bundle()
c.print_fingerprint(c.read_json("fingerprint.json"))

=== SIDIK JARI DATA ===
berkas dataset        : /opt/rumahpoint/dataset/properti_multi_kota.csv
baris mentah          : 64,650
baris bersih          : 64,228
baris latih / uji     : 51,382 / 12,846
hash baris latih      : cfe3defcae087f44fa6a7189a319ef258f4138318a7532b315bf80a85dc9990e
hash baris uji        : 718f7f914fe83904e4899da30d2fc6b7a06e249431befe26985f93e43efe7aef
hash file dataset     : 34375406a6d72b21bad55e5784caddc931a85bb20c999414d3fdd6b2f5774d18
lipatan 0 (indeks uji): f298c1b2ee29f320d674c00a96af6e48ce0dff211e66a5d6798761cf3ae938d4
lipatan 1 (indeks uji): c177e4c57ee746c9d021815cecfbc4e8128469d08d1617f086033940cee7efe6
lipatan 2 (indeks uji): 7c6dfcd5824b61717f80eeddde5371bb4c93adb3a21beddf20741479178c025e
lipatan 3 (indeks uji): 9956fdc032bc252aeb7d0ecac5531626bac860dee609478a49110408ee8b19cd
lipatan 4 (indeks uji): a0b04506e601150298462828e0306e977fd83ad22063aa0ebd10d9663b7c51dc


=== SIDIK JARI DATA ===
berkas dataset        : /opt/rumahpoint/dataset/properti_multi_kota.csv
baris mentah          : 64,650
baris bersih          : 64,228
baris latih / uji     : 51,382 / 12,846
hash baris latih      : cfe3defcae087f44fa6a7189a319ef258f4138318a7532b315bf80a85dc9990e
hash baris uji        : 718f7f914fe83904e4899da30d2fc6b7a06e249431befe26985f93e43efe7aef
hash file dataset     : 34375406a6d72b21bad55e5784caddc931a85bb20c999414d3fdd6b2f5774d18
lipatan 0 (indeks uji): f298c1b2ee29f320d674c00a96af6e48ce0dff211e66a5d6798761cf3ae938d4
lipatan 1 (indeks uji): c177e4c57ee746c9d021815cecfbc4e8128469d08d1617f086033940cee7efe6
lipatan 2 (indeks uji): 7c6dfcd5824b61717f80eeddde5371bb4c93adb3a21beddf20741479178c025e
lipatan 3 (indeks uji): 9956fdc032bc252aeb7d0ecac5531626bac860dee609478a49110408ee8b19cd
lipatan 4 (indeks uji): a0b04506e601150298462828e0306e977fd83ad22063aa0ebd10d9663b7c51dc


## 2. Muat hasil

Semua angka di bawah berasal dari CSV/JSON yang ditulis notebook sebelumnya.

In [3]:
wajib = [
    "regression_results_holdout.csv",
    "regression_results_cv.csv",
    "classification_results_holdout.csv",
    "classification_results_cv.csv",
    "regression_summary.json",
    "classification_summary.json",
    "age_feature_test.json",
    "fingerprint.json",
]
hilang = [nama for nama in wajib if not (c.OUTPUT_DIR / nama).exists()]
assert not hilang, f"Hasil belum lengkap: {hilang}. Jalankan notebook 02 dan 03 lebih dulu."

reg_holdout = c.read_csv("regression_results_holdout.csv")
reg_cv = c.read_csv("regression_results_cv.csv")
cls_holdout = c.read_csv("classification_results_holdout.csv")
cls_cv = c.read_csv("classification_results_cv.csv")
reg_summary = c.read_json("regression_summary.json")
cls_summary = c.read_json("classification_summary.json")
age_test = c.read_json("age_feature_test.json")
fp = c.read_json("fingerprint.json")
print("Berkas hasil lengkap.")

for nama, ringkasan in (("regresi", reg_summary), ("klasifikasi", cls_summary)):
    for kunci in ("train_sha256", "test_sha256"):
        assert ringkasan["sidik_jari"][kunci] == fp["split"][kunci], (
            f"Sidik jari {nama} tidak cocok dengan fingerprint.json: {kunci}"
        )
print("Sidik jari ringkasan regresi dan klasifikasi cocok dengan fingerprint.json.")

Berkas hasil lengkap.
Sidik jari ringkasan regresi dan klasifikasi cocok dengan fingerprint.json.


## 3. Regresi: hold-out dan CV

Tabel hold-out:

In [4]:
display(reg_holdout[[
    "model", "mae_rupiah", "rmse_rupiah", "r2_log10", "r2_rupiah",
    "mape_pct", "fit_seconds", "predict_seconds", "model_size_mb",
]].style.format({
    "mae_rupiah": "Rp {:,.0f}",
    "rmse_rupiah": "Rp {:,.0f}",
    "r2_log10": "{:.4f}",
    "r2_rupiah": "{:.4f}",
    "mape_pct": "{:.2f}%",
    "fit_seconds": "{:.2f}",
    "predict_seconds": "{:.3f}",
    "model_size_mb": "{:.2f}",
}))

,model,mae_rupiah,rmse_rupiah,r2_log10,r2_rupiah,mape_pct,fit_seconds,predict_seconds,model_size_mb
0,Linear Regression,"Rp 69,961,665,995","Rp 5,663,928,547,146",0.4634,-1607405.7889,470.14%,15.97,0.285,0.02
1,Random Forest,"Rp 581,266,918","Rp 2,974,447,350",0.8134,0.5567,83.88%,208.83,1.852,319.44
2,Gradient Boosting,"Rp 779,751,623","Rp 3,369,880,236",0.7217,0.4310,108.10%,138.44,0.719,0.15
3,XGBoost,"Rp 679,679,796","Rp 3,161,594,155",0.7887,0.4992,85.41%,14.31,0.304,0.33
4,KNN,"Rp 733,343,934","Rp 3,344,529,741",0.7500,0.4395,94.45%,4.37,25.290,172.50


Tabel CV (rata-rata ± simpangan baku):

In [5]:
display(reg_cv[[
    "model", "mae_rupiah_mean", "mae_rupiah_std", "r2_log10_mean",
    "r2_log10_std", "r2_rupiah_mean", "mape_pct_mean",
    "fit_seconds_mean", "predict_seconds_mean",
]].style.format({
    "mae_rupiah_mean": "Rp {:,.0f}",
    "mae_rupiah_std": "Rp {:,.0f}",
    "r2_log10_mean": "{:.4f}",
    "r2_log10_std": "{:.4f}",
    "r2_rupiah_mean": "{:.4f}",
    "mape_pct_mean": "{:.2f}%",
    "fit_seconds_mean": "{:.2f}",
    "predict_seconds_mean": "{:.3f}",
}))
print("Pemenang regresi:", reg_summary["winner"]["model"])
print("Alasan:", reg_summary["winner"]["reason"])

,model,mae_rupiah_mean,mae_rupiah_std,r2_log10_mean,r2_log10_std,r2_rupiah_mean,mape_pct_mean,fit_seconds_mean,predict_seconds_mean
0,Linear Regression,"Rp 99,516,982,830","Rp 68,703,917,636",0.4848,0.0116,-2671325.0037,449.68%,6.40,0.082
1,Random Forest,"Rp 631,521,340","Rp 15,836,991",0.8153,0.0071,0.5721,87.71%,204.45,1.948
2,Gradient Boosting,"Rp 832,958,586","Rp 16,235,802",0.7299,0.0085,0.3922,102.19%,99.61,0.563
3,XGBoost,"Rp 716,432,422","Rp 18,688,480",0.7958,0.0078,0.4904,93.01%,7.82,0.105
4,KNN,"Rp 795,848,652","Rp 19,500,318",0.7531,0.0091,0.4038,89.40%,3.83,15.669


Pemenang regresi: Random Forest
Alasan: Rata-rata R2 log10 CV tertinggi (0.8153) dengan MAE CV Rp 631,521,340; hold-out R2 log10 0.8134 dan MAE Rp 581,266,918.


## 4. Klasifikasi: hold-out dan CV

In [6]:
display(cls_holdout[[
    "model", "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "f1_weighted", "roc_auc", "fit_seconds", "predict_seconds", "model_size_mb",
]].style.format({
    "accuracy": "{:.4f}",
    "precision_macro": "{:.4f}",
    "recall_macro": "{:.4f}",
    "f1_macro": "{:.4f}",
    "f1_weighted": "{:.4f}",
    "roc_auc": "{:.4f}",
    "fit_seconds": "{:.2f}",
    "predict_seconds": "{:.3f}",
    "model_size_mb": "{:.2f}",
}))
display(cls_cv[[
    "model", "accuracy_mean", "accuracy_std", "f1_macro_mean", "f1_macro_std",
    "precision_macro_mean", "recall_macro_mean", "roc_auc_mean",
    "fit_seconds_mean", "predict_seconds_mean",
]].style.format({
    "accuracy_mean": "{:.4f}",
    "accuracy_std": "{:.4f}",
    "f1_macro_mean": "{:.4f}",
    "f1_macro_std": "{:.4f}",
    "precision_macro_mean": "{:.4f}",
    "recall_macro_mean": "{:.4f}",
    "roc_auc_mean": "{:.4f}",
    "fit_seconds_mean": "{:.2f}",
    "predict_seconds_mean": "{:.3f}",
}))
print("Baseline akurasi kelas mayoritas:", f"{cls_summary['baseline_akurasi_kelas_mayoritas']:.4f}")
print("Pemenang klasifikasi:", cls_summary["winner"]["model"])
print("Alasan:", cls_summary["winner"]["reason"])

,model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,roc_auc,fit_seconds,predict_seconds,model_size_mb
0,Logistic Regression,0.7769,0.7868,0.7789,0.7757,0.7754,0.8666,10.94,2.975,0.01
1,Random Forest,0.8641,0.8641,0.8640,0.8640,0.8641,0.9376,54.62,6.492,144.64
2,Gradient Boosting,0.7949,0.7955,0.7954,0.7949,0.7949,0.8734,192.56,0.694,0.15
3,XGBoost,0.8325,0.8326,0.8328,0.8325,0.8325,0.9157,11.93,1.602,0.28
4,KNN,0.8346,0.8346,0.8347,0.8346,0.8346,0.9046,8.23,38.404,172.50


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,precision_macro_mean,recall_macro_mean,roc_auc_mean,fit_seconds_mean,predict_seconds_mean
0,Logistic Regression,0.7851,0.0022,0.7841,0.0023,0.7927,0.7864,0.8700,7.99,0.239
1,Random Forest,0.8648,0.0026,0.8647,0.0026,0.8649,0.8647,0.9382,34.56,2.519
2,Gradient Boosting,0.8026,0.0059,0.8026,0.0059,0.8029,0.8029,0.8830,93.76,0.595
3,XGBoost,0.8376,0.0037,0.8376,0.0037,0.8376,0.8377,0.9208,8.02,0.247
4,KNN,0.8316,0.0025,0.8316,0.0025,0.8316,0.8316,0.9028,4.23,21.276


Baseline akurasi kelas mayoritas: 0.5085
Pemenang klasifikasi: Random Forest
Alasan: Rata-rata F1-macro CV tertinggi (0.8647), ROC-AUC CV 0.9382; hold-out akurasi 0.8641 dan F1-macro 0.8640.


## 5. Keputusan fitur usia

Kriteria sudah ditetapkan di notebook regresi sebelum eksperimen. Tabel ini
menampilkan dua varian Random Forest dan selisihnya.

In [7]:
usia_rows = []
for label in ("tanpa_usia", "dengan_usia"):
    h = age_test["results"][label]["holdout"]
    v = age_test["results"][label]["cv"]
    usia_rows.append({
        "varian": label,
        "r2_log10_holdout": h["r2_log10"],
        "mae_rupiah_holdout": h["mae_rupiah"],
        "r2_log10_cv": v["r2_log10_mean"],
        "mae_rupiah_cv": v["mae_rupiah_mean"],
    })
display(pd.DataFrame(usia_rows).style.format({
    "r2_log10_holdout": "{:.4f}",
    "mae_rupiah_holdout": "Rp {:,.0f}",
    "r2_log10_cv": "{:.4f}",
    "mae_rupiah_cv": "Rp {:,.0f}",
}))
print("Delta:", age_test["delta"])
print("Keputusan:", age_test["keputusan"])
print("Alasan:", age_test["alasan"])

,varian,r2_log10_holdout,mae_rupiah_holdout,r2_log10_cv,mae_rupiah_cv
0,tanpa_usia,0.8134,"Rp 581,266,918",0.8153,"Rp 631,521,340"
1,dengan_usia,0.8134,"Rp 583,170,875",0.8153,"Rp 634,606,654"


Delta: {'r2_holdout': -1.2055886796202842e-05, 'r2_cv': 4.127871396886196e-05, 'mae_rupiah_holdout': 1903957.2874720097, 'mae_rupiah_cv': 3085314.210693598}
Keputusan: dibuang
Alasan: Minimal satu ambang tidak terpenuhi; usia tidak dipakai sebagai fitur utama.


## 6. Perbandingan dengan jurnal

| Rujukan | Pemenang di jurnal | Catatan |
| --- | --- | --- |
| Fitri (2023), JACOST 4(1) | Random Forest | 2.011 rumah, satu wilayah, regresi harga |
| Sari & Prabowo (2025), Intellect 4(1) | XGBoost | 1.642 entri Yogyakarta, regresi harga |
| Hayya’ dkk. (2025), Sainteks 22(1) | Random Forest | 2.020 baris, KNN vs Random Forest |

Ketiganya tidak sepakat. Perbandingan ini mengulang pertanyaan yang sama pada
data yang lebih besar dan berasal dari banyak kota, dengan protokol identik
untuk lima model. Karena itu hasil kita tidak harus sama dengan pemenang jurnal;
yang penting adalah data, split, praproses, dan metrik diperlakukan sama untuk
semua kandidat.

Tabel berikut menyandingkan pemenang jurnal dengan pemenang eksperimen ini;
kolom `sama` dihitung kode, bukan diketik.


In [8]:
jurnal = pd.DataFrame(reg_summary["jurnal"])
jurnal["pemenang_rumahpoint"] = reg_summary["winner"]["model"]
jurnal["sama"] = jurnal["pemenang"] == jurnal["pemenang_rumahpoint"]
display(jurnal[["nama", "DOI", "pemenang", "pemenang_rumahpoint", "sama"]])


,nama,DOI,pemenang,pemenang_rumahpoint,sama
0,"Fitri (2023), JACOST 4(1)",10.52158/jacost.v4i1.491,Random Forest,Random Forest,True
1,"Sari & Prabowo (2025), Intellect 4(1)",10.57255/intellect.v4i1.1385,XGBoost,Random Forest,False
2,"Hayya' dkk. (2025), Sainteks 22(1)",10.30595/sainteks.v22i1.25998,Random Forest,Random Forest,True


## 7. Batas hasil

- Semua model memakai parameter bawaan; tidak ada tuning. Model yang menang di
  sini belum tentu menang setelah tuning.
- Dataset berasal dari scraping dan mengandung baris duplikat yang dilaporkan,
  bukan dibuang otomatis.
- Sejumlah nama wilayah tidak ada di aset dan tidak bisa dipilih pengguna.
- Usia bangunan hanya diuji sebagai bukti tambahan, bukan fitur utama.
- Ambang Mahal/Murah memakai median harga per kota pada seluruh data bersih,
  termasuk baris uji, mengikuti `database/seed.py`; label tidak sepenuhnya
  out-of-sample.
- Hasil hold-out dan CV dihitung pada data yang sama; bukan validasi eksternal.
- Waktu dan ukuran model bergantung pada mesin; bukan jaminan performa produksi.
- Model produksi di `cloud_ml/model/` tidak diubah.

## 8. Tulis kesimpulan

Berkas ini merangkum pemenang, tabel, keputusan usia, dan sidik jari. Semua
angka diambil dari hasil yang sudah dibaca.

In [9]:
kesimpulan = {
    "dibuat_pada": datetime.now(timezone.utc).isoformat(),
    "sidik_jari": {
        "raw_rows": fp["raw_rows"],
        "clean_rows": fp["clean_rows"],
        "n_train": fp["split"]["n_train"],
        "n_test": fp["split"]["n_test"],
        "train_sha256": fp["split"]["train_sha256"],
        "test_sha256": fp["split"]["test_sha256"],
    },
    "regresi": {
        "pemenang": reg_summary["winner"]["model"],
        "alasan": reg_summary["winner"]["reason"],
        "holdout": reg_holdout.to_dict("records"),
        "cv": reg_cv.to_dict("records"),
    },
    "klasifikasi": {
        "pemenang": cls_summary["winner"]["model"],
        "alasan": cls_summary["winner"]["reason"],
        "holdout": cls_holdout.to_dict("records"),
        "cv": cls_cv.to_dict("records"),
    },
    "usia": {
        "keputusan": age_test["keputusan"],
        "delta": age_test["delta"],
        "kriteria": age_test["criteria"],
    },
    "jurnal": reg_summary["jurnal"],
    "catatan": [
        "Semua model memakai data, fitur, praproses, split, lipatan, dan metrik yang sama.",
        "Tidak ada hyperparameter tuning.",
        "Usia bangunan tidak dipakai di perbandingan utama.",
        "Ambang Mahal/Murah memakai median per kota pada seluruh data bersih, termasuk baris uji.",
        "Hasil bukan validasi eksternal dan tidak mengubah model produksi.",
    ],
}
c.save_json("kesimpulan.json", kesimpulan)
print("Kesimpulan ditulis ke", c.OUTPUT_DIR / "kesimpulan.json")

[tulis] /opt/rumahpoint/docs/perbandingan_output/kesimpulan.json


Kesimpulan ditulis ke /opt/rumahpoint/docs/perbandingan_output/kesimpulan.json
